In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Volumes/Projets_Dev/Projets_Dev/Data_Analyst_Portfolio/Projet_1_Analyse_Ventes")
sql_path = PROJECT_ROOT / "sql" / "init_duckdb.sql"

con = duckdb.connect()
con.execute(f"SET file_search_path='{PROJECT_ROOT.as_posix()}';")
con.execute(sql_path.read_text())

**(R1) Nombre de commandes total + livrées**

In [ ]:
con.execute("""
SELECT*
FROM orders;
""").fetchdf()



In [ ]:

con.execute("""
SELECT
    COUNT(*) AS total_orders
FROM orders
WHERE order_status = 'delivered';""").fetchdf()

***(R2) Chiffre d’affaires total (commandes livrées)***

In [ ]:
con.execute("""
SELECT*
FROM order_items ;
""").fetchdf()

In [ ]:
CA_TOTAL = con.execute("""
SELECT
     SUM(A.price + A.freight_value) AS revenue
FROM order_items AS A
JOIN orders AS B
ON B.order_id = A.order_id
WHERE B.order_status = 'delivered';


""").fetchdf()

CA_TOTAL

***R3 — Chiffre d’affaires par mois***

In [ ]:
CA_MOIS =con.execute("""
SELECT
     strftime(B.order_purchase_timestamp, '%Y-%m') AS month,
     SUM(A.price + A.freight_value) AS revenue,
     COUNT(DISTINCT B.order_id) AS total_orders
FROM order_items AS A
JOIN orders AS B
   ON B.order_id = A.order_id
WHERE B.order_status = 'delivered'
GROUP BY 1
ORDER BY 1 ASC;

""").fetchdf()
CA_MOIS

***R4 — Panier moyen (AOV) des commandes livrées***

In [ ]:
AOV = con.execute("""
SELECT
     ROUND(

    SUM(oi.price + oi.freight_value) / COUNT(DISTINCT o.order_id),2) AS avg_order_value
FROM orders o
Join order_items oi ON  o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
""").fetchdf()
AOV

***R5 — Top 10 produits par chiffre d’affaires***

In [ ]:
con.execute("""
SELECT
      P.product_id AS product_id,
      SUM(O.price + O.freight_value) AS revenue,
      COUNT(*) AS items_sold
FROM products AS P
JOIN order_items AS O
   ON O.product_id = P.product_id
JOIN orders AS OD
   ON OD.order_id = O.order_id
   WHERE OD.order_status = 'delivered'
GROUP BY
            1
ORDER BY 2 DESC;

""").fetchdf()

***R6 — Top 10 catégories par chiffre d’affaires***

In [ ]:
con.execute("""
SELECT
      COALESCE(P.product_category_name, 'unknown') AS product_category_name,
      SUM(O.price + O.freight_value) AS revenue,
      COUNT(*) AS items_sold
FROM products AS P
JOIN order_items AS O
   ON O.product_id = P.product_id
JOIN orders AS OD
   ON OD.order_id = O.order_id
   WHERE OD.order_status = 'delivered'
GROUP BY
            1
ORDER BY 2 DESC;

""").fetchdf()

***R7 — Satisfaction : note moyenne + % mauvais avis***

In [ ]:
con.execute("""
SELECT
     ROUND(avg(review_score), 2) AS avg_score,
     COUNT(*) AS total_reviews,
     ROUND((SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 2) AS bad_review_rate
FROM reviews""").fetchdf()

****R8 — Produits les mieux notés vs les moins bien notés (min 50 avis)***

In [ ]:
con.execute("""
SELECT
      O.product_id AS product_id,
      ROUND(AVG(R.review_score), 2) AS avg_score,
      COUNT(*) AS reviews_count
FROM reviews AS R
JOIN order_items AS O
    ON R.order_id = O.order_id
JOIN orders AS ORD
   ON ORD.order_id = O.order_id
WHERE order_status = 'delivered'
GROUP BY O.product_id
HAVING COUNT(*)  >= 50
ORDER BY AVG(R.review_score) desc
LIMIT 10;


""").fetchdf()




In [ ]:
con.execute("""
SELECT
      O.product_id AS product_id,
      ROUND(AVG(R.review_score), 2) AS avg_score,
      COUNT(*) AS reviews_count
FROM reviews AS R
JOIN order_items AS O
    ON R.order_id = O.order_id
JOIN orders AS ORD
   ON ORD.order_id = O.order_id
WHERE order_status = 'delivered'
GROUP BY O.product_id
HAVING COUNT(*)  >= 50
ORDER BY AVG(R.review_score) ASC
LIMIT 10;


""").fetchdf()